# AI Co-Scientist: Testing Hypotheses
# SOLUTIONS

Solutions to the challenges in `lessons/3_Testing_Hypotheses.ipynb`. Run the setup cells first.

In [ ]:
# Imports and paths used in this notebook.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'scripts' / '03_tests'))
import hypothesis_tests as ht
from evaluator import load_rows
from reveal import G_ACCEPTED, implied_g, load_key

In [ ]:
rows = load_rows(ROOT / 'data' / 'observations.csv')
train = [row for row in rows if row['split'] == 'train']
test = [row for row in rows if row['split'] == 'test']
train_logs = ht.logs(train)
test_logs = ht.logs(test)
all_logs = ht.logs(rows)

## 🥊 Challenge 1: Add a Fifth Hypothesis

In [ ]:
params = ht.fit([], (2.0, -1.0), train_logs)
test_error = np.mean(np.abs(ht.residuals(params, test_logs)))
print(f'H5 held-out error: {test_error:.3f} dex')

In [ ]:
boot = ht.bootstrap(rows, np.random.default_rng(ht.SEED))
for how in ('rows', 'groups'):
    p_low, p_high = np.percentile(boot[how]['p'], [2.5, 97.5])
    q_low, q_high = np.percentile(boot[how]['q'], [2.5, 97.5])
    print(f'resampling {how:7}: p in [{p_low:.3f}, {p_high:.3f}]   q in [{q_low:.3f}, {q_high:.3f}]')

## 🥊 Challenge 2: Plot the Bootstrap

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(boot['groups']['q'], bins=50)
ax.axvline(-0.5, color='black', linestyle='--', label='q = -1/2')
ax.set_xlabel('q (exponent of x2)')
ax.set_ylabel('bootstrap draws')
ax.legend()
plt.show()

In [ ]:
key = load_key(ROOT / 'data' / 'key.json')
for name, meaning in key['meaning'].items():
    print(f'{name} = {meaning}')
print(key['groups'])

In [ ]:
log_c_h4 = ht.fit([], (1.5, -0.5), all_logs)[0]
g_estimate = implied_g(10 ** log_c_h4, key)
print(f'G = {g_estimate:.3e} (accepted {G_ACCEPTED:.3e}, {100 * (g_estimate / G_ACCEPTED - 1):+.1f}%)')

## 🥊 Challenge 3: Which Interval Contains G?

In [ ]:
for how in ('rows', 'groups'):
    ends = np.percentile(boot[how]['log10_c_h4'], [2.5, 97.5])
    low, high = sorted(implied_g(10 ** value, key) for value in ends)
    inside = low <= G_ACCEPTED <= high
    print(f'resampling {how:7}: G in [{low:.3e}, {high:.3e}], contains accepted value: {inside}')